In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from load import load_data, tensorize_data, clean_data
from models import NaiveRNN, NaiveLSTM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
def prepareDataLoader(dataset, labels, batchsize=32):
    train, test, train_labels, test_labels = train_test_split(dataset, labels, test_size=0.2, random_state=42)


    train_dataset = TensorDataset(train, train_labels)
    test_dataset = TensorDataset(test, test_labels)

    train_dataloader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

    return (train_dataloader, test_dataloader)

In [3]:
def training(epochs, model, dataloader, criterion, optimizer):
    for epoch in range(epochs):
        model.train()
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

def evaluate(model, dataloader):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Accuracy: {100 * correct / total}%')


In [4]:
dfs = load_data()

# The load_data function will return dfs in a different order. Parse through
for df in dfs:
    if df[1] == "PFC_con_4.csv":
        con4_df = df[0]
    if df[1] == "PFC_con_5.csv":
        con5_df = df[0]

In [5]:
# Use DS+/DS- as labels (0)
tensors_4 = tensorize_data(con4_df, 0)
labels = tensors_4[1]
dataset = tensors_4[0]

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

In [6]:
rnn_model = NaiveRNN(1, 32, 2).to(device)
lstm_model = NaiveLSTM(1, 32, 2).to(device)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)


epochs = 50
training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/50], Loss: 0.6966
Epoch [2/50], Loss: 0.6957
Epoch [3/50], Loss: 0.6977
Epoch [4/50], Loss: 0.7191
Epoch [5/50], Loss: 0.6757
Epoch [6/50], Loss: 0.6854
Epoch [7/50], Loss: 0.7053
Epoch [8/50], Loss: 0.6761
Epoch [9/50], Loss: 0.6845
Epoch [10/50], Loss: 0.6912
Epoch [11/50], Loss: 0.6771
Epoch [12/50], Loss: 0.6985
Epoch [13/50], Loss: 0.6891
Epoch [14/50], Loss: 0.6827
Epoch [15/50], Loss: 0.6974
Epoch [16/50], Loss: 0.6837
Epoch [17/50], Loss: 0.6944
Epoch [18/50], Loss: 0.6911
Epoch [19/50], Loss: 0.6750
Epoch [20/50], Loss: 0.6829
Epoch [21/50], Loss: 0.7039
Epoch [22/50], Loss: 0.7000
Epoch [23/50], Loss: 0.7140
Epoch [24/50], Loss: 0.6841
Epoch [25/50], Loss: 0.6895
Epoch [26/50], Loss: 0.6872
Epoch [27/50], Loss: 0.6767
Epoch [28/50], Loss: 0.6810
Epoch [29/50], Loss: 0.6899
Epoch [30/50], Loss: 0.7104
Epoch [31/50], Loss: 0.6723
Epoch [32/50], Loss: 0.6913
Epoch [33/50], Loss: 0.7259
Epoch [34/50], Loss: 0.7063
Epoch [35/50], Loss: 0.7042
Epoch [36/50], Loss: 0.6847
E

In [7]:
con4_df_clean = clean_data(con4_df)
# Use DS+/DS- as labels (0)
dataset, labels = tensorize_data(con4_df_clean, 0)

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

epochs = 20
rnn_model = NaiveRNN(1, 32, 2).to(device)
training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.7076
Epoch [2/20], Loss: 0.6633
Epoch [3/20], Loss: 0.7330
Epoch [4/20], Loss: 0.6939
Epoch [5/20], Loss: 0.7157
Epoch [6/20], Loss: 0.6999
Epoch [7/20], Loss: 0.7076
Epoch [8/20], Loss: 0.6989
Epoch [9/20], Loss: 0.6955
Epoch [10/20], Loss: 0.6807
Epoch [11/20], Loss: 0.7078
Epoch [12/20], Loss: 0.6988
Epoch [13/20], Loss: 0.7265
Epoch [14/20], Loss: 0.7891
Epoch [15/20], Loss: 0.7291
Epoch [16/20], Loss: 0.6745
Epoch [17/20], Loss: 0.7047
Epoch [18/20], Loss: 0.7284
Epoch [19/20], Loss: 0.6403
Epoch [20/20], Loss: 0.6770
Accuracy: 49.12349045578496%


In [10]:
ds_minus = con4_df_clean[con4_df_clean.iloc[:, 3] == 0]
dataset, labels = tensorize_data(ds_minus, 0)

unsqueezed_tensor = dataset.unsqueeze(-1)

train, test = prepareDataLoader(unsqueezed_tensor, labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.01)
epochs = 20
rnn_model = NaiveRNN(1, 32, 2).to(device)


training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.7147
Epoch [2/20], Loss: 0.6997
Epoch [3/20], Loss: 0.7484
Epoch [4/20], Loss: 0.6390
Epoch [5/20], Loss: 0.7029
Epoch [6/20], Loss: 0.7351
Epoch [7/20], Loss: 0.6520
Epoch [8/20], Loss: 0.6818
Epoch [9/20], Loss: 0.7382
Epoch [10/20], Loss: 0.6914
Epoch [11/20], Loss: 0.7165
Epoch [12/20], Loss: 0.7507
Epoch [13/20], Loss: 0.6489
Epoch [14/20], Loss: 0.7680
Epoch [15/20], Loss: 0.7240
Epoch [16/20], Loss: 0.6860
Epoch [17/20], Loss: 0.6901
Epoch [18/20], Loss: 0.6694
Epoch [19/20], Loss: 0.7245
Epoch [20/20], Loss: 0.7062
Accuracy: 51.77249707830152%


In [9]:
ds_minus = con4_df_clean[con4_df_clean.iloc[:, 3] == 0]
dataset, labels = tensorize_data(ds_minus, 0)
